# **Semana 12: Bases de Datos Vectoriales y RAG (NB1 - Conceptual)**

## **Propósito de la Sesión**
Implementar búsqueda semántica y flujo RAG (Retrieval Augmented Generation) utilizando ChromaDB como base de datos vectorial y sentence-transformers para generar embeddings. Este es el fundamento de los sistemas modernos de preguntas y respuestas basados en conocimiento privado.

### **Objetivos de Aprendizaje**
Al finalizar este notebook, el estudiante será capaz de:
1. **Comprender** el concepto de embeddings y su rol en la búsqueda semántica.
2. **Instalar y configurar** ChromaDB, una base de datos vectorial ligera.
3. **Generar embeddings** de frases usando `sentence-transformers`.
4. **Almacenar vectores** en ChromaDB junto con metadatos.
5. **Realizar búsquedas por similitud** (k-NN) sobre los embeddings.
6. **Entender** el flujo RAG y cómo LangChain lo orquesta.

## **Configuración Inicial**

Instalaremos las librerías necesarias: chromadb para la base de datos vectorial, sentence-transformers para los embeddings, y langchain (opcional) para entender la orquestación RAG.

In [ ]:
# Instalación de librerías
!pip install chromadb sentence-transformers --quiet
!pip install langchain langchain-community --quiet  # Opcional, para referencia RAG

# Importaciones
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
from sklearn.decomposition import PCA
import random

# Configuración de visualización
sns.set_style("whitegrid")
pd.set_option('display.max_colwidth', 200)

print("Librerías instaladas e importadas correctamente.")

---
## **Parte 1: Conceptos Fundamentales**

### **¿Qué es un embedding?**

Un **embedding** es una representación vectorial (lista de números) de un objeto (texto, imagen, audio) en un espacio de alta dimensionalidad. Los vectores se generan de manera que objetos semánticamente similares tengan vectores cercanos según alguna métrica de distancia.

Por ejemplo, las frases "El gato duerme en el sofá" y "Un felino descansa en el sillón" tendrán vectores similares, aunque no compartan palabras exactas.

### **¿Qué es una base de datos vectorial?**

Una base de datos vectorial está optimizada para:
*   Almacenar vectores junto con metadatos.
*   Indexar los vectores para búsquedas rápidas de vecinos cercanos (ANN - Approximate Nearest Neighbors).
*   Calcular similitudes (coseno, euclidiana, producto punto).

### **¿Qué es RAG (Retrieval Augmented Generation)?**

RAG es un patrón arquitectónico que combina:
1. **Retriever**: Busca documentos relevantes en una base de conocimiento (vectorial).
2. **Generator**: Un LLM (Large Language Model) genera respuestas basadas en los documentos recuperados.

Flujo típico:
1. Indexación: Documentos → Fragmentos → Embeddings → Base vectorial.
2. Consulta: Pregunta → Embedding → Búsqueda de fragmentos similares.
3. Generación: Prompt (pregunta + fragmentos) → LLM → Respuesta.

---
## **Parte 2: Generación de Embeddings con Sentence Transformers**

Utilizaremos el modelo `all-MiniLM-L6-v2` de Sentence Transformers, que genera embeddings de 384 dimensiones. Es pequeño, rápido y adecuado para prototipos.

In [ ]:
# Cargar el modelo
print("Cargando modelo de embeddings...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print(f"Modelo cargado. Dimensionalidad: {model.get_sentence_embedding_dimension()}")

# Frases de ejemplo
frases = [
    "El gato duerme en el sofá",
    "Un perro juega en el parque",
    "Me encanta la comida italiana",
    "El cielo está nublado hoy",
    "Los felinos son animales independientes",
    "El canino corre detrás de la pelota",
    "La pasta con salsa es deliciosa",
    "Mañana lloverá según el pronóstico",
    "Los gatos domésticos requieren cuidados",
    "El adiestramiento canino requiere paciencia"
]

# Generar embeddings
print("\nGenerando embeddings...")
embeddings = model.encode(frases)
print(f"Embeddings generados: {embeddings.shape}")

# Mostrar un ejemplo
print("\nPrimeros 10 valores del embedding de la primera frase:")
print(embeddings[0][:10])
print(f"... y {len(embeddings[0])-10} valores más.")

### **Visualización de embeddings (PCA)**

Aunque no podemos visualizar 384 dimensiones, podemos reducirlas a 2D con PCA para ver cómo se agrupan las frases semánticamente similares.

In [ ]:
# Reducir dimensionalidad a 2D con PCA
pca = PCA(n_components=2)
embeddings_2d = pca.fit_transform(embeddings)

# Crear DataFrame para visualización
df_viz = pd.DataFrame({
    'x': embeddings_2d[:, 0],
    'y': embeddings_2d[:, 1],
    'frase': frases,
    'tema': ['gato']*3 + ['perro']*2 + ['comida']*2 + ['clima']*2 + ['gato']*1 + ['perro']*1  # Aproximación manual
})

plt.figure(figsize=(10, 8))
sns.scatterplot(data=df_viz, x='x', y='y', hue='tema', style='tema', s=100)
for i, row in df_viz.iterrows():
    plt.annotate(f"{i}", (row['x'], row['y']), fontsize=9, alpha=0.7)
plt.title('Visualización 2D de Embeddings de Frases (PCA)')
plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 2')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("Observa cómo las frases sobre gatos (0,4,8) y perros (1,5,9) tienden a agruparse.")

---
## **Parte 3: ChromaDB - Base de Datos Vectorial**

ChromaDB es una base de datos vectorial ligera y fácil de usar, ideal para prototipos y proyectos pequeños. Almacena documentos, embeddings y metadatos.

In [ ]:
# Crear cliente de Chroma (en memoria para este ejemplo)
client = chromadb.Client(Settings(
    chroma_db_impl="duckdb+parquet",
    persist_directory="./chroma_db"  # Para persistir en disco
))

# O crear cliente efímero (solo en memoria)
# client = chromadb.EphemeralClient()

print("Cliente ChromaDB creado.")

### **3.1. Crear una colección**

Las colecciones en Chroma son como tablas en SQL. Almacenan documentos, embeddings y metadatos.

In [ ]:
# Crear una colección
collection_name = "frases_demo"

# Si ya existe, la eliminamos para empezar limpio
try:
    client.delete_collection(collection_name)
except:
    pass

collection = client.create_collection(
    name=collection_name,
    metadata={"hnsw:space": "cosine"}  # Métrica de similitud (coseno)
)

print(f"Colección '{collection_name}' creada.")

### **3.2. Insertar documentos con embeddings**

Chroma puede generar los embeddings automáticamente si le proporcionamos una función de embedding, pero nosotros ya los tenemos. Insertaremos:
*   `ids`: Identificadores únicos.
*   `documents`: El texto original.
*   `embeddings`: Los vectores que generamos.
*   `metadatas`: Información adicional (opcional).

In [ ]:
# Preparar datos para inserción
ids = [f"frase_{i}" for i in range(len(frases))]
metadatas = [{"tema": "gato" if i in [0,4,8] else "perro" if i in [1,5,9] else "comida" if i in [2,6] else "clima"} for i in range(len(frases))]

# Insertar en Chroma
collection.add(
    ids=ids,
    documents=frases,
    embeddings=embeddings.tolist(),  # Chroma espera listas, no arrays numpy
    metadatas=metadatas
)

print(f"Insertados {len(frases)} documentos en ChromaDB.")

# Verificar el conteo
print(f"Total en colección: {collection.count()}")

---
## **Parte 4: Búsqueda por Similitud en ChromaDB**

Ahora realizaremos consultas a la base de datos vectorial. Probaremos con diferentes frases de búsqueda.

In [ ]:
def buscar_frases(query, n_results=3):
    """Función para buscar frases similares a la consulta"""
    print(f"\n🔍 Consulta: '{query}'")
    
    # Generar embedding de la consulta
    query_embedding = model.encode([query]).tolist()[0]
    
    # Buscar en Chroma
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        include=["documents", "metadatas", "distances"]
    )
    
    # Mostrar resultados
    print(f"\nResultados (similitud de coseno, mayor = más similar):")
    for i, (doc, metadata, distance) in enumerate(zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0]
    )):
        # En Chroma, distance = 1 - cosine_similarity (para cosine)
        similarity = 1 - distance
        print(f"  {i+1}. [Sim: {similarity:.4f}] {doc} (tema: {metadata.get('tema', 'N/A')})")
    
    return results

# Probar diferentes consultas
buscar_frases("animales domésticos que duermen en casa")
buscar_frases("comida que me gusta")
buscar_frases("clima y pronóstico del tiempo")
buscar_frases("mascotas peludas")

---
## **Parte 5: Búsqueda con Filtros (Metadatos)**

Chroma permite filtrar por metadatos antes de la búsqueda vectorial. Por ejemplo, buscar solo frases sobre "gatos".

In [ ]:
def buscar_con_filtro(query, filtro, n_results=3):
    """Búsqueda con filtro por metadatos"""
    print(f"\n🔍 Consulta: '{query}' con filtro {filtro}")
    
    query_embedding = model.encode([query]).tolist()[0]
    
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        where=filtro,  # Filtro estilo Chroma: {"tema": "gato"}
        include=["documents", "metadatas", "distances"]
    )
    
    print(f"\nResultados filtrados:")
    for i, (doc, metadata, distance) in enumerate(zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0]
    )):
        similarity = 1 - distance
        print(f"  {i+1}. [Sim: {similarity:.4f}] {doc} (tema: {metadata.get('tema', 'N/A')})")
    
    return results

# Buscar solo entre frases de tema 'gato'
buscar_con_filtro("animales que ronronean", {"tema": "gato"})

# Buscar solo entre frases de tema 'perro'
buscar_con_filtro("animales que ladran", {"tema": "perro"})

---
## **Parte 6: Introducción a RAG con LangChain (Simulación)**

Aunque no ejecutaremos un LLM real en este notebook (requiere API key), simularemos el flujo RAG para entender cómo LangChain orquesta el proceso.

In [ ]:
# Simulación de RAG
def flujo_rag_simulado(pregunta):
    print("="*60)
    print(f"📝 PREGUNTA: {pregunta}")
    print("="*60)
    
    # Paso 1: Generar embedding de la pregunta
    print("\n1. Generando embedding de la pregunta...")
    query_emb = model.encode([pregunta]).tolist()[0]
    
    # Paso 2: Buscar documentos relevantes en Chroma
    print("2. Buscando documentos relevantes en ChromaDB...")
    results = collection.query(
        query_embeddings=[query_emb],
        n_results=2,
        include=["documents", "distances"]
    )
    
    documentos_recuperados = results['documents'][0]
    distancias = results['distances'][0]
    
    print(f"   Documentos recuperados:")
    for i, (doc, dist) in enumerate(zip(documentos_recuperados, distancias)):
        sim = 1 - dist
        print(f"     • [Similitud: {sim:.4f}] {doc}")
    
    # Paso 3: Construir el prompt con contexto
    print("\n3. Construyendo prompt para el LLM con los documentos recuperados...")
    contexto = "\n".join([f"- {doc}" for doc in documentos_recuperados])
    prompt = f"""Responde la pregunta basándote en el siguiente contexto.

Contexto:
{contexto}

Pregunta: {pregunta}
Respuesta:"""
    print("\n📋 PROMPT GENERADO:")
    print(prompt)
    
    # Paso 4: Simular respuesta del LLM
    print("\n4. LLM genera respuesta basada en el contexto...")
    # Simulación: Elegimos una respuesta plausible según los documentos
    if any("gato" in doc.lower() for doc in documentos_recuperados):
        respuesta_simulada = "Los gatos son animales independientes que suelen dormir en sofás y requieren cuidados como alimentación y caja de arena."
    elif any("perro" in doc.lower() for doc in documentos_recuperados):
        respuesta_simulada = "Los perros son animales juguetones que necesitan ejercicio diario y adiestramiento con paciencia."
    elif any("comida" in doc.lower() for doc in documentos_recuperados):
        respuesta_simulada = "La comida italiana, como la pasta, es muy apreciada por su sabor y variedad de salsas."
    else:
        respuesta_simulada = "El clima actual indica posibilidad de lluvia según el pronóstico meteorológico."
    
    print(f"🤖 RESPUESTA SIMULADA: {respuesta_simulada}")
    
    return respuesta_simulada

# Probar el flujo RAG simulado
flujo_rag_simulado("¿Cómo son los gatos?")
print("\n")
flujo_rag_simulado("¿Qué me dices de los perros?")

---
## **Ejercicios Prácticos para el Estudiante**

Ahora aplica lo aprendido con estos ejercicios.

### **Ejercicio 1: Crear una colección con tus propias frases**

Crea una nueva colección en Chroma llamada `mis_frases` con al menos 5 frases sobre temas que te interesen (deportes, música, tecnología, etc.). Genera los embeddings con el mismo modelo, inserta los documentos y realiza una búsqueda semántica.

In [ ]:
# --- INICIO DE TU CÓDIGO ---

# 1. Crear nueva colección
# ...

# 2. Definir frases
# ...

# 3. Generar embeddings
# ...

# 4. Insertar en Chroma
# ...

# 5. Realizar una búsqueda
# ...

# --- FIN DE TU CÓDIGO ---

### **Ejercicio 2: Comparar métricas de distancia**

Chroma por defecto usa distancia coseno (cuando se configura `"hnsw:space": "cosine"`). Investiga cómo cambiar la métrica a "l2" (euclidiana) y crea una nueva colección con esa métrica. Inserta los mismos documentos y compara los resultados para una misma consulta. ¿Qué observas?

In [ ]:
# --- INICIO DE TU CÓDIGO ---

# Crear colección con métrica L2
# ...

# Insertar documentos
# ...

# Realizar misma consulta en ambas colecciones
# ...

# Comparar
# ...

# --- FIN DE TU CÓDIGO ---

### **Ejercicio 3: RAG con un documento más largo**

Simula un documento más largo (por ejemplo, un párrafo sobre historia) y divídelo en fragmentos. Genera embeddings para cada fragmento, almacénalos en Chroma y prueba un flujo RAG con una pregunta sobre ese documento.

*Pista: Puedes usar `text_splitter` de LangChain para dividir el texto.*

In [ ]:
# --- INICIO DE TU CÓDIGO ---

# Texto largo de ejemplo
texto_largo = """
La Revolución Industrial fue un período de transformación tecnológica, 
económica y social que comenzó en Gran Bretaña a finales del siglo XVIII 
y se extendió por Europa y América durante el siglo XIX. 
Se caracterizó por la invención de máquinas como la máquina de vapor, 
el desarrollo de fábricas y la producción en masa. 
Esto llevó a cambios profundos en la agricultura, la manufactura, 
la minería, el transporte y la tecnología. 
La Revolución Industrial también tuvo un impacto significativo en 
las condiciones sociales, con el crecimiento de ciudades industriales 
y la aparición de nuevas clases sociales como el proletariado.
"""

# Dividir en fragmentos (simulado)
# ...

# Indexar en Chroma
# ...

# Preguntar
# ...

# --- FIN DE TU CÓDIGO ---

### **Ejercicio 4: Reflexión sobre casos de uso**

En una celda Markdown, describe un caso de uso real para bases de datos vectoriales y RAG en uno de los siguientes dominios:
*   Atención al cliente (chatbot de soporte)
*   Comercio electrónico (búsqueda de productos)
*   Educación (tutor inteligente)
*   Legal (búsqueda en jurisprudencia)

Explica qué datos se indexarían, cómo se generarían los embeddings y qué preguntas respondería el sistema.

---
## **Conclusiones**

En esta sesión hemos:
1. **Comprendido** los conceptos de embeddings, bases de datos vectoriales y RAG.
2. **Generado** embeddings de frases usando `sentence-transformers`.
3. **Almacenado** vectores en ChromaDB, una base de datos vectorial ligera.
4. **Realizado** búsquedas por similitud semántica, con y sin filtros.
5. **Simulado** un flujo RAG para entender cómo se combina recuperación y generación.

**Conceptos clave:**
*   Los embeddings capturan el significado semántico.
*   Las bases de datos vectoriales permiten búsqueda por similitud a gran escala.
*   RAG mejora las respuestas de LLMs con conocimiento externo y actualizado.
*   LangChain y otros orquestadores simplifican la implementación de RAG.

En el próximo notebook, aplicaremos estos conceptos con Pinecone (base vectorial en la nube) y LangChain para construir un sistema RAG completo.